In [1]:
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langchain-text-splitters==1.1.2" "python-dotenv==1.2.3"


Note: you may need to restart the kernel to use updated packages.


<h3>개념 요약</h3>
<p>RAG의 생성 단계는 검색한 문서와 질문을 함께 모델에 전달합니다. 출처 태그를 문맥에 포함하고 답에도 같은 형식으로 표시하도록 요청합니다.</p>

In [2]:
import os

from dotenv import find_dotenv, load_dotenv

env_file = find_dotenv(usecwd=True)
assert env_file and load_dotenv(env_file), '프로젝트 루트 .env를 찾지 못했습니다.'
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY가 없습니다.'
print('환경 준비 완료')


환경 준비 완료


<h3>예제 실행</h3>
<p>답과 출처가 분명한 문서 두 건을 실제 임베딩으로 색인합니다.</p>

In [3]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings

docs = [Document(page_content='정기 휴관일은 매주 월요일입니다.',
                 metadata={'doc_id': 'D01', 'page': 1}),
        Document(page_content='대출 기간은 14일입니다.',
                 metadata={'doc_id': 'D02', 'page': 2})]
store = InMemoryVectorStore.from_documents(docs, OpenAIEmbeddings(model='text-embedding-3-small'))


<h3>검색 결과로 답 생성</h3>
<p>검색한 문서의 ID와 쪽 번호를 문맥에 넣고 실제 생성 모델에 전달합니다.</p>

In [4]:
question = '정기 휴관일은 언제인가요?'
hits = store.similarity_search(question, k=1)
context = '\n'.join(
    f"[{d.metadata['doc_id']} p.{d.metadata['page']}] {d.page_content}" for d in hits
)
prompt = f'문서만 근거로 답하고 출처를 붙이세요.\n[문서]\n{context}\n[질문] {question}'
answer = init_chat_model('openai:gpt-5.4-mini', temperature=0).invoke(prompt)
print(answer.text)


정기 휴관일은 **매주 월요일**입니다. [D01 p.1]


<h3>변형 실행</h3>
<p><code>k</code>를 2로 바꿨을 때 모델에게 전달되는 문맥이 어떻게 늘어나는지 확인합니다.</p>

In [5]:
more_hits = store.similarity_search(question, k=2)
print([d.metadata['doc_id'] for d in more_hits])


['D01', 'D02']


<h3>일부러 실패해 보기</h3>
<p>쪽 번호를 기록하지 않은 문서는 출처 태그를 만들 수 없습니다. 실제 <code>KeyError</code>를 포착합니다.</p>

In [6]:
broken = Document(page_content='휴관일은 월요일입니다.', metadata={'doc_id': 'D01'})
try:
    print(f"[{broken.metadata['doc_id']} p.{broken.metadata['page']}]")
except KeyError as error:
    print('예상한 실패: 누락된 메타데이터', error)


예상한 실패: 누락된 메타데이터 'page'


<h3>실행 검증</h3>
<p>앞 셀의 결과가 이 절에서 확인할 조건을 만족하는지 단언문으로 검사합니다.</p>

In [7]:
assert '월요일' in answer.text
assert 'D01' in answer.text and 'p.1' in answer.text
print('검증 통과: 실제 생성 응답에 답과 출처가 있습니다.')


검증 통과: 실제 생성 응답에 답과 출처가 있습니다.


<h3>정리</h3>
<p>출처 표기는 생성 단계에서 갑자기 생기지 않습니다. 로더가 남긴 메타데이터를 검색 결과와 함께 프롬프트에 전달해야 답의 근거를 확인할 수 있습니다.</p>